# Section 4 · Notebook 2 — Agent as Tool: Tutoring Assistant (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file.

A teacher agent receives a student's question and delegates to the right specialist tutor (math, writing, science). The teacher always stays in control.

In [4]:
%pip install -q -r requirements-groq.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os

from dotenv import load_dotenv
from strands import Agent
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

# --- FORCE ENVIRONMENT CLEAN UP ---
# This wipes out any hidden, cached environment variables causing the 404
os.environ.pop("GROQ_BASE_URL", None)
os.environ.pop("GROQ_LLM_MODEL", None)
os.environ.pop("GROQ_MODEL_ID", None)

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq")

demo_agent = Agent(model=model)
response = demo_agent("Hello, Groq! what is special today?")
print()
print("-" * 60)
print(response)

Using openai/gpt-oss-120b on Groq
Hey there! 🎉 It’s October 25, so let’s take a quick tour of what makes today a little extra‑special in the world of history, culture, and celebrations.

---

## 📅  Historical Highlights (What’s happened on Oct 25)

| Year | Event |
|------|-------|
| **1415** | **Battle of Agincourt** – Henry V’s English army defeats the French in one of the most famous battles of the Hundred Years’ War. |
| **1854** | **“The Great Gold Rush”** – The first gold strike in the **Kootenay region of British Columbia** (Canada) sparks a mini‑boom. |
| **1917** | **The **Russian **October Revolution** (according to the Julian calendar) officially begins; the Bolsheviks seize the Winter Palace in Petrograd. |
| **1945** | **United Nations** – The **UN Charter** comes into effect, establishing the world’s first global intergovernmental organization devoted to peace and security. |
| **1971** | **UN General Assembly** adopts **Resolution 2758**, recognizing the People’s Republi

In [6]:
from strands import Agent

## Specialist agents

Each specialist gets a `name` and a `description`. When an agent is used as a tool, the name becomes the tool name and the description tells the teacher when to call it.

In [7]:
math_tutor = Agent(
    name="math_tutor",
    description="Solves mathematical equations and explains math concepts step by step",
    model=model,
    system_prompt="You are a math tutor. Solve math problems and explain your solutions clearly, step by step.",
    callback_handler=None,
)

writing_coach = Agent(
    name="writing_coach",
    description="Helps with writing, grammar, and essay structure",
    model=model,
    system_prompt="You are a writing coach. Help with writing, grammar, and essay structure. Provide constructive feedback.",
    callback_handler=None,
)

science_helper = Agent(
    name="science_helper",
    description="Explains scientific concepts and principles in simple terms",
    model=model,
    system_prompt="You are a science tutor. Explain scientific concepts clearly using simple language and real-world examples.",
    callback_handler=None,
)

## The teacher (orchestrator)

Pass the specialist agents straight into `tools=[...]` — Strands wraps each one as a callable tool.

In [8]:
teacher = Agent(
    model=model,
    system_prompt="""You are a teacher who helps students by delegating to specialist tutors.
Never solve problems yourself. Always delegate to the appropriate specialist:
- math_tutor: for math problems and equations
- writing_coach: for writing, grammar, and essays
- science_helper: for science questions

After receiving the specialist's response, present it to the student in a friendly way.""",
    tools=[math_tutor, writing_coach, science_helper],
)

## Test with different questions

In [9]:
# Math → math_tutor
response = teacher("Can you help me solve this quadratic equation: x^2 + 5x + 6 = 0?")


Tool #1: math_tutor
Sure thing! Let’s walk through solving the quadratic equation  

\[
x^{2}+5x+6=0
\]

step‑by‑step.

---

### 1️⃣ Try factoring first
A quadratic \(ax^{2}+bx+c\) can often be written as \((x+p)(x+q)=0\) if we can find two numbers \(p\) and \(q\) that satisfy:

* **Sum**: \(p+q = b\) (the coefficient of \(x\))
* **Product**: \(p\cdot q = c\) (the constant term)

For \(x^{2}+5x+6\):

| Want | Value |
|------|-------|
| Sum  | \(p+q = 5\) |
| Product | \(p\cdot q = 6\) |

Looking at pairs that multiply to 6:

* \(1 \times 6 = 6\) → sum 7 (no)
* **\(2 \times 3 = 6\)** → sum 5 (yes!)

So \(p=2\) and \(q=3\).  
That means  

\[
x^{2}+5x+6 = (x+2)(x+3)
\]

Set each factor to zero (Zero‑Product Property):

\[
\begin{cases}
x+2 = 0 \;\Rightarrow\; x = -2\\[4pt]
x+3 = 0 \;\Rightarrow\; x = -3
\end{cases}
\]

**Solutions:** \(x = -2\) or \(x = -3\).

---

### 2️⃣ Verify with the quadratic formula (optional)
The quadratic formula works for any \(ax^{2}+bx+c=0\):

\[
x = \frac{-

In [10]:
# Writing → writing_coach
response = teacher("How do I write a strong thesis statement for my essay about climate change?")


Tool #2: writing_coach
Absolutely! A strong thesis is the backbone of your essay—it tells your reader exactly what you’ll argue and how you’ll support that argument. Below is a step‑by‑step guide, some ready‑made examples, and a quick checklist to help you craft a compelling thesis for your climate‑change essay.

---

## 1. What Makes a Thesis “Strong”?

| Element | Why It Matters | How to Write It |
|---------|----------------|-----------------|
| **Clear, debatable claim** | Shows you’re taking a stance that someone could argue against. | Use strong verbs like *argues, contends, asserts, demonstrates.* |
| **Specific focus** | Keeps the essay manageable and shows depth. | Mention a region, sector, time frame, or policy (e.g., *U.S. transportation*, *by 2030*). |
| **Roadmap of main points** | Gives readers a preview of the evidence you’ll present. | List 2‑4 supporting ideas in parallel structure (e.g., *because it reduces emissions, creates jobs, and protects vulnerable communities

In [ ]:
# Science → science_helper
response = teacher("Why does ice float on water?")

OpenAI threw rate limit error



Tool #3: science_helper


OpenAI threw rate limit error
OpenAI threw rate limit error
OpenAI threw rate limit error
OpenAI threw rate limit error
OpenAI threw rate limit error
